In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.stats import t

np.random.seed(42)

In [3]:
def CI_mean(means, confidence=0.95):
    n = len(means)
    mean = np.mean(means)
    t_test_statistic = t.ppf(1 - 0.05/2, df=n-1)
    S_n = np.std(means, ddof=1)
    lower_bound = mean - t_test_statistic * S_n / np.sqrt(n)
    upper_bound = mean + t_test_statistic * S_n / np.sqrt(n)
    return mean,lower_bound, upper_bound

In [4]:
## Exercise 1:
n = 100
U = np.random.rand(1000)
X_i = np.e ** (U)
theta_MC = np.mean(X_i)

print("Point estimate:", theta_MC)
print("Confidence Interval of theta MC", CI_mean(X_i)[1:])
print("Analytical Solution:", np.e -1)

Point estimate: 1.7034864447751372
Confidence Interval of theta MC (np.float64(1.6727142143120217), np.float64(1.7342586752382527))
Analytical Solution: 1.718281828459045


In [5]:
## Exercise 4:
sub_intervals = 10
lower_interval = np.arange(0, 1.0, 1/sub_intervals)
upper_interval = np.arange(0.1, 1.1, 1/sub_intervals)

intervals = np.array([lower_interval, upper_interval]).T

U_stratisfied = np.zeros((n//sub_intervals,sub_intervals))

for j in range(1, sub_intervals+1):
    U_stratisfied[:, j-1] = np.random.uniform(((j-1)/sub_intervals), (j/sub_intervals), n//sub_intervals)

Y_i = np.mean(np.exp(U_stratisfied), axis=1)
theta_SS = np.mean(Y_i)

print("Point estimate:", theta_SS)
print("Confidence Interval of theta MC", CI_mean(Y_i)[1:])
print("Analytical Solution:", np.e -1)
    

Point estimate: 1.7225271793809331
Confidence Interval of theta MC (np.float64(1.7115993633715454), np.float64(1.7334549953903209))
Analytical Solution: 1.718281828459045


## Exercise 5

In [6]:
class BlockingSystem():
    def __init__(self, m_service_time=8, m_arriving_time=1, num_servers=10, num_customers=10*10000):
        self.lamd = 1/m_arriving_time # mean inter-arrival time
        self.mu = 1/m_service_time # mean service time
        self.num_servers = num_servers
        self.num_customers = num_customers
        
        self.servers_available = self.num_servers # m
        self.servers_status = self.servers_status = np.zeros(self.num_servers)
        self.blocked = 0
        
    def sample_service_time(self): # ~ Exp(mu)
        U = np.random.rand()
        return -1* np.log(U) /self.mu 

    def sample_arrival_time(self): # ~ Exp(lamd)
        U = np.random.rand()
        return -1 * np.log(U) /self.lamd 
    
    
    def allocate_server(self, arrival_time, service_time):
        servers_available = np.sum(self.servers_status <= arrival_time) # Number of servers that are available at the time of arrival
        
        if servers_available > 0:
            bool = True
            allocated_server = np.argmin(self.servers_status)
            self.servers_status[allocated_server] = arrival_time + service_time
            
        else:
            bool = False
        return bool
    
    def blocking_probability(self):
        return self.blocked / ( self.num_customers)    
    
    def analytical_blocking_probability(self): 
        A = self.lamd / self.mu # lambda * s
        numerator = A**self.num_servers / math.factorial(self.num_servers)
        denominator = np.sum([A**i / math.factorial(i) for i in range(self.num_servers + 1)])
        
        
        return numerator / denominator # B = P(m)
    
    def reset(self):
        self.servers_status = np.zeros(self.num_servers)
        self.blocked = 0
        
    
    def simulate(self):
        t = 0
        c = 0
        self.event_array = np.zeros((self.num_customers, 2))
        arrival_times = np.zeros(self.num_customers)
        self.service_times = np.zeros(self.num_customers)
        
        for c in range(self.num_customers): # Each iteration corresponds to one customer
            
            
            customer = np.zeros(2) #arrival time, departure time
            
            c_arrival_time = self.sample_arrival_time()
            arrival_times[c] = c_arrival_time
            customer[0] = t + c_arrival_time
            service_time = self.sample_service_time()
            

            if self.allocate_server(customer[0], service_time):
                customer[1] = customer[0] + service_time
                self.service_times[c] = service_time
                
            else:
                customer[1] = -1
                self.blocked += 1
                self.service_times[c] = -1
                
            self.event_array[c, :] = customer
            
            
            
            t += c_arrival_time
            
    def plot_A_t(self, ax=None, t_max=10):
        if ax is None:
            ax = plt.gca()
        count = np.arange(1, self.num_customers + 1)
        ax.plot(self.event_array[:, 0], count, label='Arrivals')
        ax.plot(np.sort(self.event_array[:, 1][self.event_array[:, 1] != -1]), count[self.event_array[:, 1] != -1], label='Departures', c='green')
        ax.plot(self.event_array[:, 0][self.event_array[:, 1] == -1], count[:len(self.event_array[:, 0][self.event_array[:, 1] == -1])], label='Rejections', c='red')
        
        
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Event Count')
        ax.set_xlim(0, t_max)
        ax.set_ylim(0, self.event_array[:, 0][self.event_array[:, 0] <= t_max].shape[0]*1.1)
        ax.set_title(f'Events Over Time with $\\mu={self.mu}$, $\\lambda={self.lamd}$, $m={self.num_servers}$')
        ax.legend()
        ax.grid()
        #plt.show()
    
    def plot_inter_times(self, ax=None):
        if ax is None:
            ax = plt.gca()
        inter_arrival_times = np.diff(self.event_array[:, 0])
        inter_departure_times = np.diff(np.sort(self.event_array[:, 1][self.event_array[:, 1] != -1]))
        
        ax.hist(inter_arrival_times, bins=50, density=True, alpha=0.6, color='g', label='Inter-arrival Times')
        ax.hist(inter_departure_times, bins=50, density=True, alpha=0.6, color='b', label='Inter-departure Times')
        
        ax.set_title('Inter-arrival and Inter-departure Times Distribution')
        ax.set_xlabel('Time')
        ax.set_ylabel('Density')
        ax.grid()
        ax.legend()
        plt.show()
            
            

In [14]:
num_simulations = 100
X_i = np.zeros(num_simulations)
blocking_system = BlockingSystem()

Z_i = np.zeros(num_simulations)

for i in range(num_simulations):
    blocking_system.reset()
    blocking_system.simulate()
    X_i[i] = blocking_system.blocking_probability()
    arrival_times = np.diff(blocking_system.event_array[:, 0])
    Z_i[i] = np.mean(arrival_times)
    
c = -1 *np.cov(X_i, Z_i, ddof=1)[0,1] / np.var(Z_i, ddof=1)
Y_i = X_i + c*(Z_i - 1/blocking_system.lamd)
print(f"c = {c:.3f}")

theta = np.mean(X_i)
var_theta = np.var(X_i, ddof=1)
print(f"Theta hat: {theta:.4f}")
print(f"Theta (analytical probability): {blocking_system.analytical_blocking_probability():.4f}")


theta_SS = np.mean(Y_i)
var_theta_SS = np.var(Y_i, ddof=1)
print(f"Theta_SS hat: {theta_SS:.4f}")

t_test_statistic = t.ppf(1 - 0.05/2, df=num_simulations-1)
CI = (theta - t_test_statistic*np.sqrt(var_theta)/np.sqrt(num_simulations), theta + t_test_statistic*np.sqrt(var_theta)/np.sqrt(num_simulations))

print(f"95% CI Theta hat: [{CI[0]:.4f}, {CI[1]:.4f}]")

t_test_statistic = t.ppf(1 - 0.05/2, df=num_simulations-1)
CI = (theta_SS - t_test_statistic*np.sqrt(var_theta_SS)/np.sqrt(num_simulations), theta_SS + t_test_statistic*np.sqrt(var_theta_SS)/np.sqrt(num_simulations))

print(f"95% CI Theta SS hat: [{CI[0]:.4f}, {CI[1]:.4f}]")



c = 0.335
Theta hat: 0.1215
Theta (analytical probability): 0.1217
Theta_SS hat: 0.1217
95% CI Theta hat: [0.1212, 0.1218]
95% CI Theta SS hat: [0.1214, 0.1219]


In [16]:
print(f"Variance reduction: {(var_theta_SS - var_theta)*100 / var_theta}%")

Variance reduction: -28.50195016038008%
